In [1]:
from sklearn.linear_model import LinearRegression
from bokeh.models import HoverTool 
from bokeh.io import export_svg
import numpy as np
import transforms3d as td
import pandas as pd
import bokeh.io
import bokeh.plotting
# Import libraries
from mpl_toolkits import mplot3d
import matplotlib.pyplot as plt
import re
import panel as pn
from bokeh.models import ColumnDataSource
from bokeh.palettes import Viridis
bokeh.io.output_notebook()
pn.extension()
import plotly.graph_objects as go
from bokeh.io import export_png
import holoviews as hv
from bokeh.models import CategoricalColorMapper, Legend

Loading BokehJS ...

In [121]:
p1_opens = {'passive_start' : [[66, 110], [199, 236], [305, 348], [425, 462], [535, 578], [658, 702], [769, 810], [875, 920], [994, 1030], [1090, 1135], [1204, 1249]],
'active_rorcr' : [[1856, 1934], [2505, 2575], [3244, 3305]],
'active_cube' : [[626, 680], [1003, 1053], [1946, 1990], [3327, 3371], [4834, 4878], [5978, 6028], [6178, 6219], [6549, 6590], [6853, 6892], [7260, 7303], [7550, 7590], [9115, 9161], [9322, 9366], [9788, 9831], [10697, 10746], [11063, 11108], [16324, 16369], [17175, 17220], [17741, 17782], [18209, 18249], [18661, 18699], [23887, 23910], [24319, 24342], [24440, 24504], [26049, 26097], [26274, 26304]],
'passive_end' : [[225, 266], [319, 351], [414, 455], [501, 541], [593, 632], [705, 743], [805, 844], [896, 936], [995, 1044], [1099, 1139]]}

p13_opens = {'passive_start' : [[350, 385], [572, 600], [752, 786], [906, 934], [1045, 1075], [1180, 1209], [1312, 1345], [1448, 1478], [1571, 1599], [1695, 1728]],
             'active_rorcr' : [[2924, 2943], [4940, 4955], [5354, 5377], [6881, 6895]],
             'active_cube' : [[3483, 3499], [5476, 5491], [6269, 6283], [7012, 7041], [7473, 7498], [8209, 8227], [8489, 8502], [8679, 8692], [9599, 9620], [10555, 10569], [10628, 10647], [12731, 12747]],
             'passive_end' : [[144, 167], [337, 358], [505, 521], [686, 702], [862, 883], [1035, 1052], [1206, 1223], [1353, 1374], [1523, 1538], [1668, 1689]]}

p3_opens = {'passive_start': [[150, 196], [384, 432], [568, 606], [751, 794], [928, 963], [1097, 1142], [1271, 1311], [1442, 1483], [1592, 1631], [1751, 1804]],
            'active_rorcr': [[1868, 1922], [2724, 2753], [3503, 3532]],
            'active_cube' : [[2588, 2635], [5546, 5601], [7733, 7788], [17792, 17809], [18952, 18966], [20984, 21007]],
            'passive_end' : [[87, 142], [263, 324], [469, 528], [662, 703], [854, 899], [1050, 1091], [1232, 1281], [1419, 1454], [1590, 1633], [1766, 1807], [1942, 1982]]}


p3_old_opens = {'passive_start' : [[316, 371], [810, 866], [1029, 1080], [1228, 1282], [1427, 1476], [1623, 1679], [1805, 1864], [1979, 2030], [2119, 2178], [2272, 2329]],
            'active_rorcr' : [[6454, 6523], [7650, 7753], [9392, 9444]],
            'active_cube' : [[2328, 2696], [4601, 4651], [5807, 5861], [14749, 14812], [15108, 15186]],
            'passive_end' : [[146, 198], [552, 610], [858, 913], [1099, 1148], [1273, 1327], [1486, 1537], [1681, 1736], [1885, 1936], [2055, 2111], [2253, 2303]]}

opens_dict = {'p1' : p1_opens, 'p13': p13_opens, 'p3' : p3_opens}


In [225]:
class PlotData(object):
    def __init__(self, patient, opens):
        self.patient = patient
        self.file = '/home/jxu/hand_orthosis_ws/src/trakstar_ros/collected_data/parsed_data/%s_parsed_data.csv'%patient
        self.avg_fit = {'passive_start' : [], 'active_rorcr' : [], 'active_cube' : [], 'passive_end': []}
        self.opens = opens
        self.conditions = opens.keys()
        self.plts = []
        self.plot_label = {'passive_start': 'P1 Phase',
                            'passive_end': 'P2 Phase',
                            'active_rorcr': 'T4 Phase',
                            'active_rorcr1': 'T4 Phase',
                            'active_cube': 'Functional Training Phase',
                            'mcp_angle': 'MCP Angle',
                            'pip_angle': 'PIP Angle',
                            'm': 'Stiffness (N/mm)',
                            'motor_position': 'Cable Retraction (mm)',
                            'futek': 'Force (N)',
                            'time_open': 'Time over Experimental Condition (unscaled)',
                            'time_unitless' : 'Time over Experiment',
                            'p1': 'S1',
                            'p13': 'S2',
                            'p3': 'S3'}
        self.get_df()
        self.get_opens()

    def p_format(self, p):
        p.title.text_font_size = '20pt'
        p.legend.label_text_font_size = '15pt'
        p.xaxis.axis_label_text_font_size = '20pt'
        p.yaxis.axis_label_text_font_size = '20pt'
        p.yaxis.major_label_text_font_size = '18pt'
        p.xaxis.major_label_text_font_size = '18pt'

        return p

    def get_df(self):
        df = pd.read_csv(self.file)
        df['time_unitless'] = df['index']
        df.set_index(['condition', 'c_index', 'index'], inplace=True)
        df['open'] = 0
        df['m'] = 0
        df['b'] = 0
        df['time_open']=0

        self.full_df = df

    def get_opens(self):
        d = 1
        for c in self.opens.keys():
            o=1
            m = 0
            b = 0
            for open in self.opens[c]:
                i = open[0]
                f = open[1]
                self.full_df.loc[(c, np.r_[i:f], slice(None)), 'open'] = o
                x_vals = self.full_df.loc[(c, np.r_[i:f], slice(None)), 'motor_position'].copy().values.reshape(-1, 1)
                y_vals = self.full_df.loc[(c, np.r_[i:f], slice(None)), 'futek'].copy().values
                model = LinearRegression()
                model.fit(x_vals, y_vals)
                m += model.coef_
                b += model.intercept_
                self.full_df.loc[(c, np.r_[i:f], slice(None)), 'm'] = np.ones(f-i)*model.coef_
                self.full_df.loc[(c, np.r_[i:f], slice(None)), 'b'] = np.ones(f-i)*model.intercept_
                self.full_df.loc[(c, np.r_[i:f], slice(None)), 'time_open'] = d
                o += 1
                d+=1
            self.avg_fit[c] = [m/(o-1), b/(o-1)]
        self.df = self.full_df[self.full_df['open']>0]
        self.df.reset_index(inplace = True)
        self.df.set_index(['condition', 'open'], inplace = True)

    def plot_mcp_pip(self, df, condition):
        x = 'c_index'
        y = 'mcp_angle'
        z = 'pip_angle'
        df = df.loc[(condition, slice(None), slice(None)), :].copy().reset_index()
        hover = HoverTool(tooltips=[("index", "$index"),])
        p = bokeh.plotting.figure(
                width=800,
                height=600,
                x_axis_label="Time over Experimental Condition (unscaled)",
                y_axis_label="Angle (degres)",
                title = self.plot_label[condition] + ' MCP and PIP angles',
                y_range = [-45, 200],
                tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'reset']
            )

        p.circle(df[x], df[y], legend_label = self.plot_label[y], alpha = 0.5)
        p.circle(df[x], df[z], color = 'red', legend_label = self.plot_label[z], alpha = 0.5)
        p = self.p_format(p)

        return p

    def plot_angles(self, df, save = False):
        for i in self.conditions:
            p = self.get_mcp_pip_plot(df, i)
            #p.legend.click_policy = 'hide'
            bokeh.io.show(p)
            if save == True:
                self.plts.append(p)

    def plot_xy(self, df, x, y, z=False, save = False):
        title = "%s vs. %s"%(self.plot_label[y], self.plot_label[x])
        if z != False:
            title = "%s %s and %s vs. %s"%(self.plot_label[self.patient], self.plot_label[y], self.plot_label[z], self.plot_label[x])
        col = bokeh.palettes.d3['Category10'][10]
        hover = HoverTool(tooltips=[("index", "$index"),])
        p = bokeh.plotting.figure(
                width=1000,
                height=600,
                x_axis_label=self.plot_label[x],
                y_axis_label=self.plot_label[y],
                title = title,
                tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'reset']
            )
        p.add_layout(Legend(), 'right')
        for i, c in enumerate(self.conditions):
            p.circle(df.loc[(c, slice(None)), x], df.loc[(c, slice(None)), y], color = col[i], legend_label = self.plot_label[c])
            if z != False: 
                p.line(df.loc[(c, slice(None)), x], df.loc[(c, slice(None)), z], color = 'grey', line_width = 3, legend_label = self.plot_label[z])

        p.legend.location='top_left'
        p.legend.click_policy='hide'
        p = self.p_format(p)
        bokeh.io.show(p)
        if save == True: 
            export_png(p, filename='%s_%s.png'%(self.plot_label[self.patient], title))
        return p
    
    def plot_stiffness(self, time_scaled = False, save = False):
        if time_scaled == True:
            x = 'time_unitless'
        else:
            x = 'time_open'
        y = 'm'
        title = "%s vs. %s"%(self.plot_label[y], self.plot_label[x])
        col = bokeh.palettes.d3['Category10'][10]
        hover = HoverTool(tooltips=[("index", "$index"),])
        p = bokeh.plotting.figure(
                width=800,
                height=600,
                x_axis_label= 'Hand Opening',
                y_axis_label=self.plot_label[y],
                title = '%s Stiffness (N/mm) for each hand opening'%self.plot_label[self.patient],
                tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'reset'],
                y_range = [0, 0.87]
            )

        for i, c in enumerate(self.conditions):
            p.circle(self.df.loc[(c, slice(None)), x], self.df.loc[(c, slice(None)), y], color = col[i], legend_label = self.plot_label[c], size = 5)
            p.segment(x0=self.df.loc[(c, slice(None)), x], x1=self.df.loc[(c, slice(None)), x], y0=0, y1=self.df.loc[(c, slice(None)), y], color = col[i], legend_label = self.plot_label[c])
        
        p.xaxis.major_label_text_color = None  # turn off x-axis tick labels leaving space
        p.xaxis.major_tick_line_color = None  # turn off x-axis major ticks
        p.xaxis.minor_tick_line_color = None  # turn off x-axis minor ticks
        p.legend.location='top_left'
        p.legend.click_policy='hide'
        p = self.p_format(p)
        bokeh.io.show(p)
        if save == True: 
            export_png(p, filename='%s_%s.png'%(self.plot_label[self.patient], 'stiffness'))
        return p

    def plot_angle_angle(self, save = False):
        x = 'mcp_angle'
        y = 'pip_angle'
        title = "%s %s vs. %s"%(self.plot_label[self.patient], self.plot_label[y], self.plot_label[x])

        col = bokeh.palettes.d3['Category10'][10]
        hover = HoverTool(tooltips=[("index", "$index"),])
        p = bokeh.plotting.figure(
                width=600,
                height=600,
                x_axis_label=self.plot_label[x],
                y_axis_label=self.plot_label[y],
                title = title,
                tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'reset'],
                x_range = [-45, 100],
                y_range = [-45, 100]
            )

        for i, c in enumerate(['passive_start', 'passive_end']):
            p.circle(self.df.loc[(c, slice(None)), x], self.df.loc[(c, slice(None)), y], color = col[i], legend_label = self.plot_label[c])

        p.legend.location='bottom_right'
        p.legend.click_policy='hide'
        p = self.p_format(p)
        bokeh.io.show(p)
        if save == True:
            export_png(p, filename='%s_%s.png'%(self.plot_label[self.patient], 'angle_angle'))
        return p

    def plot_best_fit(self, save = False):
        x = 'motor_position'
        y = 'futek'

        hover = HoverTool(tooltips=[
                ("index", "$index"),
            ])
        col = bokeh.palettes.d3['Category10'][10]
        p = bokeh.plotting.figure(
                width=800,
                height=600,
                y_axis_label=self.plot_label[y],
                x_axis_label=self.plot_label[x],
                title = '%s Force vs. Displacement during Hand Opening'%self.plot_label[self.patient],
                x_range = [0, 30],
                y_range = [0, 12], 
                tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'tap', 'reset']
            )

        for i, c in enumerate(self.conditions):
            p.circle(self.df.loc[(c, slice(None)), x], self.df.loc[(c, slice(None)), y], color = col[i], legend_label = self.plot_label[c], size = 10, alpha = 0.1, line_alpha=0)
            t = np.linspace(0, 30, 100)
            for i, c in enumerate(self.conditions):
                p.line(t, t*self.avg_fit[c][0] + self.avg_fit[c][1], color = col[i], legend_label = self.plot_label[c], line_width = 5)
            
        p=self.p_format(p)
        p.legend.location='top_left'
        p.legend.click_policy='hide'
        if save == True:
            self.plts.append(p)
            export_png(p, filename='%s_%s.png'%(self.plot_label[self.patient], 'force_disp'))
        bokeh.io.show(p)

    def plot_force_disp(self, style = 'line', best_fit = False, save = False):
        x = 'motor_position'
        y = 'futek'

        hover = HoverTool(tooltips=[
                ("index", "$index"),
            ])
        col = bokeh.palettes.d3['Category10'][10]
        p = bokeh.plotting.figure(
                width=1000,
                height=600,
                y_axis_label=self.plot_label[y],
                x_axis_label=self.plot_label[x],
                title = '%s Force vs. Cable Retraction during Hand Opening'%self.plot_label[self.patient],
                x_range = [0, 30],
                y_range = [0, 12], 
                tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'tap', 'reset']
            )

        p.add_layout(Legend(), 'right')

        for i, c in enumerate(self.conditions):
            num_opens = self.df.loc[(c, slice(None))].index.values.max()
            if style == 'scatter': 
                p.circle(self.df.loc[(c, slice(None)), x], self.df.loc[(c, slice(None)), y], color = col[i], legend_label = self.plot_label[c], size = 10, alpha = 0.1, line_alpha=0)
            elif style == 'line':
                for o in range(1, num_opens+1):
                    p.line(self.df.loc[(c, o), x], self.df.loc[(c, o), y], color = col[i], legend_label = self.plot_label[c], line_width = 3, alpha = 0.2)
        if best_fit == True:
            t = np.linspace(0, 30, 100)
            for i, c in enumerate(self.conditions):
                p.line(t, t*self.avg_fit[c][0] + self.avg_fit[c][1], color = col[i], legend_label = self.plot_label[c], line_width = 5)

        p = self.p_format(p)
        p.legend.location='bottom'
        p.legend.click_policy='hide'
        if save == True:
            self.plts.append(p)
            export_png(p, filename='%s_%s.png'%(self.plot_label[self.patient], 'force_disp_scatter'))
        bokeh.io.show(p)

    def plot_opens(self, open_set = False, save = False):
        x = 'motor_position'
        y = 'futek'
        hover = HoverTool(tooltips=[
                ("index", "$index"),
            ])
        col = bokeh.palettes.Turbo256[::-1]
        
        if open_set != False: 
            cond, o_set = open_set
            num_opens = self.df.loc[(cond, slice(None))].index.values.max()
            p = bokeh.plotting.figure(
                width=1000,
                height=600,
                y_axis_label=self.plot_label[y],
                x_axis_label=self.plot_label[x],
                title = '%s Force vs. Cable Retraction during Hand Opening (%s)'%(self.plot_label[self.patient], self.plot_label[cond]),
                #y_range = [-2, 20],
                tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'tap', 'reset']
                )
            p.add_layout(Legend(), 'right')

            for o in o_set:
                p.line(self.df.loc[(cond, o), x], self.df.loc[(cond, o), y], color = col[::int(256/num_opens-1)][o], legend_label = str(o), line_width = 3, alpha = 0.5)

            p = self.p_format(p)
            p.legend.click_policy='hide'
            legend = bokeh.models.Legend(location=(10,10))
            p.add_layout(legend, 'right')
            if save == True:
                self.plts.append(p)
            bokeh.io.show(p)
        else: 
            for c in self.conditions:
                num_opens = self.df.loc[(c, slice(None))].index.values.max()
                p = bokeh.plotting.figure(
                    width=1000,
                    height=600,
                    y_axis_label=self.plot_label[y],
                    x_axis_label=self.plot_label[x],
                    title = '%s Force vs. Cable Retraction during Hand Opening (%s)'%(self.plot_label[self.patient], self.plot_label[c]),
                    #y_range = [-2, 20],
                    tools = [hover, 'pan', 'wheel_zoom', 'box_zoom', 'tap', 'reset']
                    )
                p.add_layout(Legend(), 'right')

                for o in range(1, num_opens+1):
                    p.line(self.df.loc[(c, o), x], self.df.loc[(c, o), y], color = col[::int(256/num_opens-1)][o], legend_label = str(o), line_width = 3, alpha = 0.5)

                p = self.p_format(p)
                p.legend.click_policy='hide'
                legend = bokeh.models.Legend(location=(10,10))
                p.add_layout(legend, 'right')
                if save == True:
                    self.plts.append(p)
                    export_png(p, filename='%s_%s.png'%(self.plot_label[self.patient]))
                bokeh.io.show(p)

In [226]:
p3 = PlotData('p3', opens_dict['p3'])
p13 = PlotData('p13', opens_dict['p13'])
p1 = PlotData('p1', opens_dict['p1'])

In [188]:
p1.plot_best_fit(save = True), p3.plot_best_fit(save = True), p13.plot_best_fit(save = True)

(None, None, None)

In [ ]:
p3.plot_stiffness(save = True), p13.plot_stiffness(save = True), p1.plot_stiffness(save = True)

In [227]:
p1.plot_angle_angle(save = True), p13.plot_angle_angle(save = True), p3.plot_angle_angle(save = True)

(figure(id='bee2f4e7-a308-4c8a-a132-29136de74dd7', ...),
 figure(id='2c5cabaf-a092-464c-ac0e-93d85b73b7b0', ...),
 figure(id='0583df97-e964-4bae-b09b-569cc13f25b1', ...))

In [169]:
p1.plot_force_disp(best_fit = True)

/tmp/ipykernel_492947/1781553076.py:251: PerformanceWarning: indexing past lexsort depth may impact performance.
  p.line(self.df.loc[(c, o), x], self.df.loc[(c, o), y], color = col[i], legend_label = self.plot_label[c], line_width = 3, alpha = 0.2)


In [155]:
p3.plot_xy(p3.full_df, 'time_unitless', 'motor_position', 'futek')

figure(id='618f9060-3e95-42a7-a795-d66091e71714', ...)

In [113]:
p3.plot_angle_angle(), p1.plot_angle_angle(), p13.plot_angle_angle()

(figure(id='eda48fad-228f-424c-9c26-080a6943e3c1', ...),
 figure(id='a4001f70-5776-4319-ac59-aaf9088978df', ...),
 figure(id='53d3cab2-2a3c-4efe-ad1e-f57b0a2ca3b1', ...))

In [154]:
p3.plot_opens(open_set = ('active_cube', [5]))

/tmp/ipykernel_492947/2982122222.py:287: PerformanceWarning: indexing past lexsort depth may impact performance.
  p.line(self.df.loc[(cond, o), x], self.df.loc[(cond, o), y], color = col[::int(256/num_opens-1)][o], legend_label = str(o), line_width = 3, alpha = 0.5)


In [15]:
p3.plot_angles(p3.full_df)

In [144]:
p3.plot_opens()

/tmp/ipykernel_492947/3445862098.py:311: PerformanceWarning: indexing past lexsort depth may impact performance.
  p.line(self.df.loc[(c, o), x], self.df.loc[(c, o), y], color = col[::int(256/num_opens-1)][o], legend_label = str(o), line_width = 3, alpha = 0.5)


In [ ]:
i = 0 

for p in plts:
    export_png(p, filename='/home/jxu/hand_orthosis_ws/src/trakstar_ros/parsed_data/figures/%s_%s.png'%(plot_label[patient], i))
    i+=1